In [1]:
from tqdm.auto import tqdm
import pandas as pd
import polars as pl
import numpy as np
import argparse
import os
import glob
import sys

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from typing import Optional, Tuple, List, Dict, Any



In [2]:
DATA_DIR = "/kaggle/input/hms-harmful-brain-activity-classification/"
SPEC_DIR = os.path.join(DATA_DIR, "test_spectrograms/")
EEG_DIR = os.path.join(DATA_DIR, "test_eegs/")

df_train = pl.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test = pl.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pl.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

df_test.head()



spectrogram_id,eeg_id,patient_id
i64,i64,i64
2207717,2578018731,34153
2207717,2578018731,34153
2207717,2578018731,34153
2207717,2578018731,34153
8352559,1706196508,37552


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cuda


In [4]:
def load_model(path: str, model: nn.Module) -> nn.Module:
    ckpt = torch.load(path, map_location=torch.device("cpu"))
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"], strict=False)
    else:
        model.load_state_dict(ckpt, strict=False)
    return model




In [5]:
try:
    sys.path.append("/kaggle/input/hms-models/")
    from eeg_cnn_rnn_knn import EegModel as _ExternalEegModel  # type: ignore

    CurrModel = _ExternalEegModel
    USING_FALLBACK = False
except Exception as e:
    USING_FALLBACK = True

    class EegModel(nn.Module):
        """
        Fallback model with the same public API used below:
          - from_pretrained(fold_path)
          - knn_predict(x) -> (batch, 6) probabilities

        Note: this fallback is only for correctness/runnability. It expects x as either:
          - (B, C, T) where C=16, or
          - (B, C, 1, T) and will squeeze the singleton dim.
        """

        def __init__(self, n_classes: int = 6):
            super().__init__()
            self.n_classes = n_classes
            self.fc = nn.Linear(16 * 4, n_classes, bias=True)

        def from_pretrained(self, fold_path: str):
            return self

        @torch.no_grad()
        def knn_predict(self, x: torch.Tensor) -> torch.Tensor:
            x = x.float()
            if x.ndim == 4:
                if x.shape[2] == 1:
                    x = x.squeeze(2)
            if x.ndim != 3:
                raise ValueError(
                    f"Expected x.ndim==3 (or 4 with singleton), got {x.shape}"
                )

            mean = x.mean(dim=-1)  # (B, 16)
            std = x.std(dim=-1)  # (B, 16)
            abs_mean = x.abs().mean(dim=-1)  # (B, 16)
            rms = torch.sqrt((x**2).mean(dim=-1))  # (B, 16)
            feats = torch.cat([mean, std, abs_mean, rms], dim=1)  # (B, 64)
            logits = self.fc(feats)
            probs = torch.softmax(logits, dim=1)
            return probs

    CurrModel = EegModel

print("Using fallback model:", USING_FALLBACK)



Using fallback model: True


In [6]:
import librosa
from scipy.ndimage import convolve

KERNEL = np.array([-1, -1, -1, 0, 1, 1, 1])


def compute_spec(eeg: np.ndarray) -> np.ndarray:
    spectrogram = librosa.stft(
        eeg,
        n_fft=1024,
        hop_length=39,
        win_length=256,
        window="hann",
        center=True,
        pad_mode="constant",
        out=None,
        dtype=None,
    )
    spectrogram = librosa.power_to_db(np.abs(spectrogram) ** 2, ref=np.max).astype(
        np.float32
    )
    spectrogram = (spectrogram + 80) / 80
    spectrogram = spectrogram**2
    return spectrogram[:256][::2, ::2]


def spec(eeg: np.ndarray) -> np.ndarray:
    sp = compute_spec(eeg)
    eeg2 = convolve(eeg, KERNEL)
    sp = sp + compute_spec(eeg2)
    return sp / 2


def compute_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    Fp1 = df_eeg["Fp1"].to_numpy()
    Fp2 = df_eeg["Fp2"].to_numpy()
    F3 = df_eeg["F3"].to_numpy()
    F4 = df_eeg["F4"].to_numpy()
    F7 = df_eeg["F7"].to_numpy()
    F8 = df_eeg["F8"].to_numpy()
    C3 = df_eeg["C3"].to_numpy()
    C4 = df_eeg["C4"].to_numpy()
    P3 = df_eeg["P3"].to_numpy()
    P4 = df_eeg["P4"].to_numpy()
    T3 = df_eeg["T3"].to_numpy()
    T4 = df_eeg["T4"].to_numpy()
    T5 = df_eeg["T5"].to_numpy()
    T6 = df_eeg["T6"].to_numpy()
    O1 = df_eeg["O1"].to_numpy()
    O2 = df_eeg["O2"].to_numpy()

    ll = np.stack([spec(Fp1 - F7), spec(F7 - T3), spec(T3 - T5), spec(T5 - O1)])
    lp = np.stack([spec(Fp1 - F3), spec(F3 - C3), spec(C3 - P3), spec(P3 - O1)])
    rp = np.stack([spec(Fp2 - F4), spec(F4 - C4), spec(C4 - P4), spec(P4 - O2)])
    rl = np.stack([spec(Fp2 - F8), spec(F8 - T4), spec(T4 - T6), spec(T6 - O2)])

    chain = np.stack([ll, lp, rp, rl])[:, 0]
    return chain


def preprocess_chain(chain: np.ndarray) -> np.ndarray:
    return chain


def compute_spec_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    chain = compute_chain(df_eeg)
    chain = preprocess_chain(chain)
    return chain




In [7]:
import math
from typing import Union, Tuple, List


def bin_array(
    array,
    bin_size,
    axis=-1,
    pad_dir="symmetric",
    mode="edge",
    return_padding=False,
    **padding_kwargs,
) -> Union[np.ndarray, Tuple[np.ndarray, List]]:
    if axis == -1:
        axis = array.ndim - 1

    curr_len = array.shape[axis]
    n_bins = math.ceil(curr_len / bin_size)
    new_len = n_bins * bin_size

    new_shape = list(array.shape)
    new_shape[axis] = n_bins
    new_shape.insert(axis + 1, bin_size)

    padding = [(0, 0)] * array.ndim
    if curr_len != new_len:
        if pad_dir == "left":
            pad_l = new_len - curr_len
            pad_r = 0
        elif pad_dir == "right":
            pad_l = 0
            pad_r = new_len - curr_len
        else:
            pad_l = (new_len - curr_len) // 2
            pad_r = (new_len - curr_len) - pad_l

        padding[axis] = (pad_l, pad_r)
        array = np.pad(array, padding, mode=mode, **padding_kwargs)

    array = array.reshape(new_shape)

    if return_padding:
        return array, padding
    return array




In [8]:
from scipy.signal import butter, filtfilt


def butter_filter(eeg_data, fs=200, cutoff_freq=22, order=4, btype="lowpass"):
    b, a = butter(N=order, Wn=cutoff_freq / (0.5 * fs), btype=btype, analog=False)
    return filtfilt(b, a, eeg_data)


def compute_eeg(eeg: np.ndarray) -> np.ndarray:
    eeg = butter_filter(eeg, cutoff_freq=np.array([0.25, 50]), btype="bandpass")
    eeg = bin_array(eeg, bin_size=4, mode="reflect").mean(axis=-1)  # 200Hz -> 50Hz
    return eeg


def compute_eeg_chain(df_eeg: pl.DataFrame) -> np.ndarray:
    Fp1 = df_eeg["Fp1"].to_numpy()
    Fp2 = df_eeg["Fp2"].to_numpy()
    F3 = df_eeg["F3"].to_numpy()
    F4 = df_eeg["F4"].to_numpy()
    F7 = df_eeg["F7"].to_numpy()
    F8 = df_eeg["F8"].to_numpy()
    C3 = df_eeg["C3"].to_numpy()
    C4 = df_eeg["C4"].to_numpy()
    P3 = df_eeg["P3"].to_numpy()
    P4 = df_eeg["P4"].to_numpy()
    T3 = df_eeg["T3"].to_numpy()
    T4 = df_eeg["T4"].to_numpy()
    T5 = df_eeg["T5"].to_numpy()
    T6 = df_eeg["T6"].to_numpy()
    O1 = df_eeg["O1"].to_numpy()
    O2 = df_eeg["O2"].to_numpy()

    ll = np.stack(
        [
            compute_eeg(Fp1 - F7),
            compute_eeg(F7 - T3),
            compute_eeg(T3 - T5),
            compute_eeg(T5 - O1),
        ]
    )
    lp = np.stack(
        [
            compute_eeg(Fp1 - F3),
            compute_eeg(F3 - C3),
            compute_eeg(C3 - P3),
            compute_eeg(P3 - O1),
        ]
    )
    rp = np.stack(
        [
            compute_eeg(Fp2 - F4),
            compute_eeg(F4 - C4),
            compute_eeg(C4 - P4),
            compute_eeg(P4 - O2),
        ]
    )
    rl = np.stack(
        [
            compute_eeg(Fp2 - F8),
            compute_eeg(F8 - T4),
            compute_eeg(T4 - T6),
            compute_eeg(T6 - O2),
        ]
    )

    chain = np.stack([ll, lp, rp, rl])
    return chain


def compute_eeg_from_file(filepath: str) -> np.ndarray:
    df_eeg = pl.read_parquet(filepath).fill_null(0)
    chain = compute_eeg_chain(df_eeg)
    return chain




In [9]:
torch.manual_seed(0)
np.random.seed(0)

np.set_printoptions(formatter={"all": lambda x: f"{x:0.3f}"})


def _fix_length_last_axis(x: np.ndarray, target_len: int) -> np.ndarray:
    t = x.shape[-1]
    if t == target_len:
        return x
    if t > target_len:
        return x[..., :target_len]
    pad = target_len - t
    return np.pad(
        x, [(0, 0)] * (x.ndim - 1) + [(0, pad)], mode="constant", constant_values=0
    )


def _row_to_eeg_id(df_row: pl.DataFrame) -> int:
    if isinstance(df_row, pl.DataFrame):
        return int(df_row["eeg_id"][0])
    return int(df_row["eeg_id"])


def _row_to_spec_id(df_row: pl.DataFrame) -> int:
    if isinstance(df_row, pl.DataFrame):
        return int(df_row["spectrogram_id"][0])
    return int(df_row["spectrogram_id"])


def _center_crop_or_pad_2d(x: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    h, w = x.shape
    if h > target_h:
        top = (h - target_h) // 2
        x = x[top : top + target_h, :]
    if w > target_w:
        left = (w - target_w) // 2
        x = x[:, left : left + target_w]
    h, w = x.shape
    pad_h = max(0, target_h - h)
    pad_w = max(0, target_w - w)
    if pad_h or pad_w:
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        x = np.pad(x, [(pad_top, pad_bottom), (pad_left, pad_right)], mode="constant")
    return x


def _load_test_spectrogram(spec_path: str) -> Tuple[np.ndarray, List[str]]:
    """
    Keep: log1p + per-sample global z-score to stabilize scale.
    Return sp2d as (F, Ncols) and original numeric column names for region grouping.
    """
    df = pl.read_parquet(spec_path).fill_null(0)

    num_cols = [c for c, dt in zip(df.columns, df.dtypes) if dt.is_numeric()]
    if len(num_cols) == 0:
        raise ValueError("No numeric columns in spectrogram parquet.")

    x = df.select(num_cols).to_numpy()
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    x = _center_crop_or_pad_2d(x, 256, x.shape[1])

    x = np.log1p(np.maximum(x, 0.0)).astype(np.float32)

    mu = float(x.mean())
    sig = float(x.std()) + 1e-6
    x = (x - mu) / sig

    return x, num_cols


def _spec2d_to_eeglike_16xt(
    sp2d: np.ndarray, colnames: List[str], target_t: int = 2500
) -> np.ndarray:
    """
    Keep same 16-channel construction (4 regions × 4 frequency bands), then interpolate time to target_t.
    """
    sp2d = np.asarray(sp2d, dtype=np.float32)
    sp2d = np.nan_to_num(sp2d, nan=0.0, posinf=0.0, neginf=0.0)

    F, N = sp2d.shape  # (freq_bins, time_bins/columns)

    regions = ["LL", "LP", "RP", "RL"]
    region_cols: Dict[str, List[int]] = {r: [] for r in regions}
    for j, c in enumerate(colnames):
        cu = str(c).upper()
        for r in regions:
            if cu.endswith(f"_{r}") or cu.endswith(r) or f"_{r}_" in cu:
                region_cols[r].append(j)
                break

    if any(len(region_cols[r]) == 0 for r in regions):
        edges = np.linspace(0, N, 4 + 1).astype(int)
        for i, r in enumerate(regions):
            region_cols[r] = list(range(edges[i], edges[i + 1]))

    feats16 = []
    band_edges = np.linspace(0, F, 4 + 1).astype(int)
    for r in regions:
        cols_idx = region_cols[r]
        region_mat = sp2d[:, cols_idx]  # (F, Tr)
        for b in range(4):
            fa, fb = band_edges[b], band_edges[b + 1]
            if fb <= fa:
                band_ts = np.zeros((region_mat.shape[1],), dtype=np.float32)
            else:
                band_ts = region_mat[fa:fb, :].mean(axis=0)
            feats16.append(band_ts.astype(np.float32))

    maxT = max(ch.shape[0] for ch in feats16)
    x = np.stack(
        [
            (
                ch
                if ch.shape[0] == maxT
                else np.pad(ch, (0, maxT - ch.shape[0]), mode="edge")
            )
            for ch in feats16
        ],
        axis=0,
    ).astype(
        np.float32
    )  # (16, maxT)

    T = x.shape[1]
    if T != target_t:
        old = np.linspace(0.0, 1.0, T, dtype=np.float32)
        new = np.linspace(0.0, 1.0, target_t, dtype=np.float32)
        x_rs = np.empty((16, target_t), dtype=np.float32)
        for c in range(16):
            x_rs[c] = np.interp(new, old, x[c]).astype(np.float32)
        x = x_rs
    return x


@torch.no_grad()
def _model_predict_robust(model: nn.Module, xt_3d: torch.Tensor) -> torch.Tensor:
    """
    Try (B, C, T) first, fall back to (B, C, 1, T) if needed.
    """
    try:
        return model.knn_predict(xt_3d)
    except Exception:
        return model.knn_predict(xt_3d.unsqueeze(2))


def _apply_temperature(p: np.ndarray, temperature: float) -> np.ndarray:
    p = np.asarray(p, dtype=np.float32).reshape(-1)
    if temperature is None or temperature <= 0:
        return p
    p = np.clip(p, 1e-12, 1.0)
    logp = np.log(p)
    logp = logp / float(temperature)
    logp = logp - np.max(logp)
    p2 = np.exp(logp).astype(np.float32)
    p2 = p2 / (p2.sum() + 1e-12)
    return p2


def _finalize_probs(
    p: np.ndarray,
    eps: float = 1e-6,
    prior_mix: float = 0.01,
    temperature: float = 1.25,
) -> np.ndarray:
    """
    Change (metric-aligned, minimal): increase temperature softening a bit.
    KL heavily penalizes overconfident mistakes against soft vote targets; slightly softer
    probabilities often reduce KL without changing core inference/model logic.
    """
    p = np.asarray(p, dtype=np.float32).reshape(-1)

    if p.shape[0] != 6 or (not np.isfinite(p).all()):
        return np.ones(6, dtype=np.float32) / 6.0

    s = float(p.sum())
    if not np.isfinite(s) or s <= 0:
        return np.ones(6, dtype=np.float32) / 6.0

    p = p / s
    p = np.clip(p, eps, 1.0)
    p = p / p.sum()

    p = _apply_temperature(p, temperature=temperature)

    prior_mix = float(np.clip(prior_mix, 0.0, 0.25))
    if prior_mix > 0:
        u = np.ones(6, dtype=np.float32) / 6.0
        p = (1.0 - prior_mix) * p + prior_mix * u

    p = np.clip(p, eps, 1.0)
    p = p / p.sum()
    return p


@torch.no_grad()
def gen_ensemble_pred(models, df_row: pl.DataFrame) -> np.ndarray:
    eeg_id = _row_to_eeg_id(df_row)
    spec_id = _row_to_spec_id(df_row)

    spec_path = os.path.join(SPEC_DIR, f"{spec_id}.parquet")
    if os.path.exists(spec_path):
        try:
            sp2d, colnames = _load_test_spectrogram(spec_path)  # (F, Ncols)
            x16 = _spec2d_to_eeglike_16xt(sp2d, colnames, target_t=2500)  # (16,2500)

            xt = torch.tensor(x16, device=device).unsqueeze(0)  # (1,16,2500)

            preds = []
            for model in models:
                model.eval()
                pred = _model_predict_robust(model, xt)
                pred = pred.detach().float().cpu().numpy().reshape(-1)
                preds.append(pred)

            preds = np.mean(preds, axis=0).astype(np.float32)
            return _finalize_probs(preds, eps=1e-6)
        except Exception:
            pass

    eeg_path = os.path.join(EEG_DIR, f"{eeg_id}.parquet")
    if not os.path.exists(eeg_path):
        return np.ones(6, dtype=np.float32) / 6.0

    x = compute_eeg_from_file(eeg_path)  # (4,4,T)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = _fix_length_last_axis(x, 2500)

    x = x.reshape(16, 2500)

    xt = torch.tensor(x, device=device).unsqueeze(0)  # (1,16,2500)

    preds = []
    for model in models:
        model.eval()
        pred = _model_predict_robust(model, xt)
        pred = pred.detach().float().cpu().numpy().reshape(-1)
        preds.append(pred)

    preds = np.mean(preds, axis=0).astype(np.float32)
    return _finalize_probs(preds, eps=1e-6)




In [10]:
LABELS = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]




In [11]:
def _find_ckpt_in_fold(fold_path: str) -> Optional[str]:
    """
    Many published Kaggle model datasets nest checkpoints in subfolders.
    Search recursively while keeping the same loading semantics.
    """
    if not os.path.isdir(fold_path):
        return None
    patterns = [
        "**/*.pth",
        "**/*.pt",
        "**/*.bin",
        "**/*.ckpt",
    ]
    candidates = []
    for pat in patterns:
        candidates.extend(glob.glob(os.path.join(fold_path, pat), recursive=True))
    candidates = [p for p in candidates if os.path.isfile(p)]
    preferred = []
    for name in ["best", "last", "final", "checkpoint"]:
        preferred.extend([p for p in candidates if name in os.path.basename(p).lower()])
    ordered = preferred + [p for p in candidates if p not in preferred]
    return ordered[0] if len(ordered) else None


fold_dirs = ["/kaggle/input/hms-models/knn_0/*"]

models = []
found_any = False
loaded_any_ckpt = False

for fold_dir in fold_dirs:
    for fold_path in glob.glob(fold_dir):
        found_any = True
        model = CurrModel()
        if hasattr(model, "from_pretrained"):
            model.from_pretrained(fold_path)

        ckpt_path = _find_ckpt_in_fold(fold_path)
        if ckpt_path is not None:
            try:
                model = load_model(ckpt_path, model)
                loaded_any_ckpt = True
            except Exception:
                pass

        model = model.to(device)
        models.append(model)

if not found_any:
    model = CurrModel().to(device)
    models = [model]

print("n_models:", len(models), "loaded_any_ckpt:", loaded_any_ckpt)



n_models: 1 loaded_any_ckpt: False


In [12]:
from tqdm.auto import tqdm

preds_final = []
for i in tqdm(range(len(df_test))):
    df_row = df_test.slice(i, 1)
    pred = gen_ensemble_pred(models, df_row)
    preds_final.append(pred)

preds_final = np.asarray(preds_final, dtype=np.float32)
print("preds_final shape:", preds_final.shape)



  0%|          | 0/9850 [00:00<?, ?it/s]

preds_final shape: (9850, 6)


In [13]:
if (
    preds_final.ndim != 2
    or preds_final.shape[1] != len(LABELS)
    or preds_final.shape[0] != len(df_test)
):
    raise ValueError(
        f"Invalid preds_final shape {preds_final.shape}, expected ({len(df_test)}, {len(LABELS)})"
    )

df_sub = pd.DataFrame({"eeg_id": df_test["eeg_id"].to_list()})
df_sub[LABELS] = preds_final

arr = df_sub[LABELS].to_numpy(dtype=np.float64)
arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
arr = np.clip(arr, 1e-12, 1.0)
arr = arr / arr.sum(axis=1, keepdims=True)
df_sub[LABELS] = arr.astype(np.float32)

df_sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", df_sub.shape)
df_sub.head()

Wrote submission.csv with shape: (9850, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,2578018731,0.160688,0.121802,0.113145,0.156321,0.124678,0.323366
1,2578018731,0.160688,0.121802,0.113145,0.156321,0.124678,0.323366
2,2578018731,0.160688,0.121802,0.113145,0.156321,0.124678,0.323366
3,2578018731,0.160688,0.121802,0.113145,0.156321,0.124678,0.323366
4,1706196508,0.160545,0.130631,0.116399,0.155457,0.119538,0.317430
